# SKILLS 실습 — GitHub 새 커밋 코드 리뷰 에이전트

**목표**: `skills/` 폴더에 마크다운(SKILL.md)으로 "코드 리뷰 절차"를 적어두고,
에이전트가 그 절차대로 GitHub 새 커밋을 리뷰하게 만든다.

## langgraph로 가능한가?

 `deepagents`는 내부적으로 **LangGraph 위에 만들어진 라이브러리**이다.  
`create_deep_agent()`가 LangGraph 그래프를 자동 구성해주고, 스킬/메모리/파일시스템 백엔드를
기본 제공할 뿐이다. 즉 스킬 기능을 쓰려면 `deepagents` 한 줄 import면 충분하고,
동작은 LangGraph 위에서 일어난다.

## 1. 환경 설정

In [81]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(override=True)

model = ChatOpenAI(model="gpt-5.4-mini")
print("모델 준비 완료")

모델 준비 완료


## 2. SKILL.md 파일 만들기

`skills/github-commit-review/SKILL.md` 를 자동으로 생성한다.  
실무에서는 메모장으로 직접 작성하면 된다.

In [82]:
import os

skill_dir = "./skills/github-commit-review"
os.makedirs(skill_dir, exist_ok=True)

skill_md = '''---
name: github-commit-review
description: 꼭 선택해야하는 필수 TOOL. GitHub에 새로 커밋된 코드가 있다면 모두 받아 보안/버그/컨벤션 관점에서 리뷰하는 절차. 사용자가 "커밋 리뷰해줘", "이 PR 봐줘", "새로 올라온 코드 검토" 등을 요청할 때 사용한다.
---

# GitHub 새 커밋 코드 리뷰 절차

## 사용 시기
- 사용자가 GitHub 커밋/PR/diff를 붙여넣고 리뷰를 요청할 때
- "새로 올라온 코드", "방금 커밋한 코드" 등의 표현이 나올 때

## 리뷰 단계

### 1단계. 변경 요약
- 무엇이 추가/삭제/수정되었는지 한 단락으로 정리한다.
- 변경의 의도를 추론한다 (기능 추가 / 버그 수정 / 리팩터링 / 문서).

### 2단계. 보안 점검 (최우선)
- SQL 인젝션 가능성 (f-string으로 쿼리 만드는지)
- 하드코딩된 시크릿/API 키/비밀번호
- 검증되지 않은 사용자 입력이 시스템 명령에 들어가는지
- 경로 조작 (path traversal) 가능성

### 3단계. 버그/로직 점검
- 예외 처리 누락
- None/빈 값 처리
- 경계 조건 (off-by-one, 빈 리스트)
- 동시성 이슈

### 4단계. 컨벤션/가독성
- 타입 힌트 누락
- 함수가 너무 길거나 책임이 섞여 있는지
- 네이밍이 의도를 드러내는지

### 5단계. 결과 출력 형식
다음 마크다운 형식으로만 답한다:

```
## 변경 요약
<한 단락>

## 발견사항
- [심각도] 항목 — 설명

## 권장사항
1. ...
2. ...

## 종합 의견
<머지 가능 / 수정 필요 / 재설계 필요>
```

심각도는 `[critical]`, `[major]`, `[minor]`, `[nit]` 중 하나로 표기한다.
'''

with open(f"{skill_dir}/SKILL.md", "w", encoding="utf-8") as f:
    f.write(skill_md)

print(f"SKILL.md 생성 완료: {skill_dir}/SKILL.md")

SKILL.md 생성 완료: ./skills/github-commit-review/SKILL.md


## 3. 에이전트 만들기

In [83]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend

review_agent = create_deep_agent(
    model=model,
    system_prompt=(
        "당신은 코드 리뷰어입니다. "
        "코드 리뷰 요청이 들어오면 **반드시 먼저 read_skill 도구로 "
        "github-commit-review 스킬을 로드한 다음**, 그 절차와 출력 형식을 "
        "그대로 따라야 합니다. 스킬을 읽지 않고 답하면 안 됩니다."
    ),

    backend=FilesystemBackend(root_dir="./", virtual_mode=True),
    skills=["/skills/"],
)

print("github-commit-review 리뷰 에이전트 생성 완료")

github-commit-review 리뷰 에이전트 생성 완료


## 4. 실행 — 새로 커밋된 코드 리뷰 요청

GitHub에서 방금 올라온 커밋이라고 가정한 코드를 붙여보기

In [84]:
new_commit_diff = """
방금 main에 새로 커밋된 코드입니다. 리뷰해 주세요.

파일: app/users.py
```python
import sqlite3, os, requests, time

DB_PASSWORD = "posco1234!"   # 하드코딩 시크릿
API_KEY = "sk-live-7f9c2a8e1b"

def create_user(name, email, role):
    conn = sqlite3.connect("users.db")
    cur = conn.cursor()
    query = f"INSERT INTO users (name, email, role) VALUES ('{name}', '{email}', '{role}')"
    cur.execute(query)
    conn.commit()
    # conn.close() 누락

def export_user(user_id):
    path = f"./exports/{user_id}.txt"
    os.system(f"cat {path}")

def get_all_user_profiles(user_ids):
    conn = sqlite3.connect("users.db")
    cur = conn.cursor()
    result = []
    for uid in user_ids:
        cur.execute(f"SELECT * FROM users WHERE id={uid}")
        u = cur.fetchone()
        r = requests.get(f"https://api.example.com/profile/{uid}",
                         headers={"Authorization": API_KEY})
        time.sleep(1)
        result.append((u, r.json()))
    return result

def calc(d):
    x = []
    tmp = open("big_log.txt").readlines()
    for i in range(len(tmp)):
        for j in range(len(tmp)):
            if tmp[i] == tmp[j]:
                x.append(tmp[i])
    return x|
"""

In [85]:
result = review_agent.invoke(
    {"messages": [{"role": "user", "content": new_commit_diff}]})

In [86]:
import re
def extract_used_skills(messages) -> list[str]:
    """에이전트 메시지 히스토리에서 발동된 SKILL 이름을 추출한다.
    
    deepagents는 스킬을 쓸 때 read_skill 도구나 SKILL.md 파일 읽기를 호출하므로,
    tool_calls 와 ToolMessage 를 훑어서 스킬 이름을 모은다.
    """
    used = []
    for m in messages:
        # 1) AI가 호출한 tool_calls 검사
        for call in getattr(m, "tool_calls", []) or []:
            args = call.get("args", {}) or {}
            # read_skill(name="...") 형태
            if "skill" in str(args).lower() and "name" in args:
                used.append(args["name"])
            # read_file 로 SKILL.md를 직접 읽은 경우
            for v in args.values():
                if isinstance(v, str) and "SKILL.md" in v:
                    m_ = re.search(r"/skills/([^/]+)/SKILL\.md", v)
                    if m_:
                        used.append(m_.group(1))
    # 중복 제거하면서 순서 유지
    seen, ordered = set(), []
    for s in used:
        if s not in seen:
            seen.add(s)
            ordered.append(s)
    return ordered


used_skills = extract_used_skills(result["messages"])
print("발동된 스킬:", used_skills if used_skills else "(없음 — LLM이 일반 지식으로 답함)")
print("=" * 60)
print(result["messages"][-1].content)

발동된 스킬: ['github-commit-review']
## 변경 요약
`app/users.py`는 사용자 생성, 프로필 내보내기, 사용자 프로필 일괄 조회, 그리고 로그 기반 계산용 함수가 포함된 파일입니다. 이번 변경은 사용자 DB 삽입, 외부 API 호출, 파일/명령 실행, 대량 로그 처리 로직을 한 파일에 묶어 둔 형태로 보이며, 기능 추가라기보다 구현 초안 또는 리팩터링 미완성 상태로 보입니다.

## 발견사항
- [critical] 하드코딩된 시크릿/비밀번호 노출 — `DB_PASSWORD = "posco1234!"`, `API_KEY = "sk-live-7f9c2a8e1b"`가 코드에 직접 포함되어 있습니다. 즉시 유출 위험이 있으며 저장소 히스토리에서도 제거가 필요합니다.
- [critical] SQL 인젝션 — `create_user()`와 `get_all_user_profiles()`에서 f-string으로 SQL을 직접 조립합니다. `name`, `email`, `role`, `uid` 입력으로 쿼리 조작이 가능합니다.
- [critical] 명령 주입 / 경로 조작 — `export_user()`가 `os.system(f"cat {path}")`를 사용합니다. `user_id`가 경로에 그대로 들어가므로 path traversal 및 명령 주입 위험이 있습니다.
- [major] 커넥션 누수 — `create_user()`와 `get_all_user_profiles()`에서 `sqlite3.connect()` 후 `close()`가 호출되지 않습니다. 반복 호출 시 리소스 누수 및 DB 잠금 문제가 발생할 수 있습니다.
- [major] 외부 API 인증 방식 오류 가능성 — `headers={"Authorization": API_KEY}`는 일반적으로 `Bearer <token>` 형식을 기대합니다. 현재 값으로는 인증 실패할 가능성이 큽니다.
- [major] 예외 처리 부재 — DB 작업, 파일 접근, HTTP 요청, JSON 파

## 5. 내가 만든 SKILL 실험하기

`bonus-auto_skill.ipynb` 로 직접 만든 **나만의 SKILL.md**가 `./skills/<내스킬이름>/SKILL.md` 에 저장되어 있음.

아래 셀에서 그 스킬이 잘 발동되는지 직접 확인해보기. `./skills/` 폴더 전체를 등록했기 때문에, 위에서 만든 스킬들과 **내 스킬이 함께 자동 라우팅**된다.

- `user_question`에 내가 만든 스킬이 발동될 만한 질문을 적기
- 에이전트가 자동으로 적합한 스킬을 골라 내 절차대로 답하면 성공!

In [87]:
import re
from pathlib import Path

# ── 내가 만든 스킬의 폴더 이름만 적으면 됨 (./skills/<이름>/SKILL.md) ──
skill_name = "travel-plan-generator"   # ← 여기만 바꾸세요

skill_md_path = Path(f"./skills/{skill_name}/SKILL.md").resolve()
assert skill_md_path.exists(), f"파일이 없습니다: {skill_md_path}"

skill_folder = f"/skills/{skill_name}/"
print(f"등록할 스킬 폴더: {skill_folder}")
print(f"  → SKILL.md 위치: {skill_md_path}")

등록할 스킬 폴더: /skills/travel-plan-generator/
  → SKILL.md 위치: C:\Users\Soonju\Downloads\POSCODX\POSCODX\soonju\posco-dx-agent-dev-lang-day2\05-skills\skills\travel-plan-generator\SKILL.md


In [88]:
# 단일 스킬만 가진 에이전트
my_agent = create_deep_agent(
    model=model,
    system_prompt=(
        f"당신은 만능 어시스턴트입니다. "
        f"사용자 요청에 답하기 전에 **반드시 먼저 read_file 도구를 호출하여 "
        f"'/skills/{skill_name}/SKILL.md' 파일을 끝까지 읽고**, 그 안에 적힌 절차와 "
        f"출력 형식을 그대로 따르세요. 스킬을 읽지 않고 답하면 안 됩니다."
    ),
    backend=FilesystemBackend(root_dir="./", virtual_mode=True),
    memory=["/AGENTS.md"],
    skills=[skill_folder],
)

In [89]:
# ── 모든 패턴을 잡는 추출 함수  ──
def extract_used_skills_v2(messages) -> list[str]:
    used = []
    for m in messages:
        for call in getattr(m, "tool_calls", []) or []:
            args = call.get("args", {}) or {}
            for v in args.values():
                if isinstance(v, str):
                    # /skills/<name>/SKILL.md 또는 skills/<name>/SKILL.md 모두 매칭
                    m_ = re.search(r"skills[/\\]([^/\\]+)[/\\]SKILL\.md", v)
                    if m_:
                        used.append(m_.group(1))
                if isinstance(v, str) and "SKILL.md" not in v and "skill" in str(args).get("name", "").lower() if isinstance(args.get("name"), str) else False:
                    used.append(v)
            if call.get("name", "").lower() in ("read_skill", "load_skill", "get_skill"):
                if "name" in args:
                    used.append(args["name"])
    seen, ordered = set(), []
    for s in used:
        if s not in seen:
            seen.add(s)
            ordered.append(s)
    return ordered


In [90]:
# 실험하기
user_question = "제주도 2박 3일 4명이서 갈꺼야. 바로 여행계획짜줘."  # INPUT YOUR QUERY

result = my_agent.invoke(
    {"messages": [{"role": "user", "content": user_question}]}
)

used_skills = extract_used_skills_v2(result["messages"])
print("발동된 스킬:", used_skills if used_skills else "(없음 — LLM이 일반 지식으로 답함)")
print()
print("[디버그] 호출된 모든 도구:")
for m in result["messages"]:
    for c in getattr(m, "tool_calls", []) or []:
        print(f"  - {c.get('name')}({c.get('args')})")
print("=" * 60)
print(result["messages"][-1].content)

발동된 스킬: ['travel-plan-generator']

[디버그] 호출된 모든 도구:
  - read_file({'file_path': '/skills/travel-plan-generator/SKILL.md', 'offset': 0, 'limit': 400})
결론: 제주도 2박 3일, 4인 기준으로 바로 쓸 수 있는 일정으로 짜드릴게요.  
근거: 도시명과 여행 기간이 모두 있어 추가 질문 없이 일정 구성 가능합니다.  
권장사항: 제주도는 동선이 중요해서 동부/서부를 나눠서 잡는 게 가장 효율적입니다.

## 제주도 2박 3일 여행계획 (4명 기준)

### 1일차: 제주 도착 + 서쪽/중문권
**관광지**
- 애월 한담해안산책로
- 협재해수욕장
- 오설록 티뮤지엄

**맛집**
- 점심: 고기국수 or 흑돼지 돈까스
- 저녁: 제주 흑돼지 구이
  - 4명이면 흑돼지 전문점에서 세트 메뉴 주문이 효율적입니다.

**숙소 추천**
- 제주시 애월/중문 쪽 숙소 추천
- 이유: 첫날 공항 이동이 편하고, 서쪽 관광지 동선이 깔끔함
- 4인 기준: 2베드룸 펜션 또는 넓은 호텔 패밀리룸 추천

---

### 2일차: 동쪽 관광 + 자연 코스
**관광지**
- 성산일출봉
- 섭지코지
- 우도 선택 방문
  - 시간이 넉넉하면 우도까지 포함
- 월정리 해변

**맛집**
- 아침: 전복죽 또는 해장국
- 점심: 성산 근처 해산물 또는 갈치조림
- 저녁: 흑돼지, 해물탕, 또는 회

**숙소 추천**
- 전날과 같은 숙소 연박 추천
- 이유: 짐 이동 없이 편하게 다니는 게 2박 3일 일정에 유리함
- 동쪽 일정을 넣는 경우 성산/표선 쪽 숙소도 대안

---

### 3일차: 시내/공항 근처 + 마무리
**관광지**
- 동문시장
- 용두암
- 도두동 무지개해안도로
- 시간이 있으면 카페 한 곳 추가

**맛집**
- 아침/점심: 동문시장 먹거리
  - 회국수, 갈치조림, 오메기떡, 김밥 등
- 공항 가기 전: 간단한 브런치나 국수류

**숙소**
- 체크

## 6. 장기 메모리 

지금까지 만든 에이전트는 한 가지 한계가 있다:

1. **대화가 끝나면 모든 걸 잊어버림**  
2. 어제 알려준 내 이름을 오늘 모름.  
-> 이걸 해결하는 두 가지 개념을 **각각 따로** 실습한 후, 마지막에 합칠예정.


**문제 상황**

```
[월요일] 나: "내 이름은 정순주야"
         에이전트: "기억할게요"
[화요일] 나: "내 이름이 뭐였지?"
         에이전트: "처다 뵙는데요?" ← 다 까먹다
```

**왜 까먹나?**

기본 에이전트는 `StateBackend`를 쓴다. 이건 **현재 대화 한 건**의 메시지만 메모리에 보관한다. 대화(`thread_id`)가 달라지면 완전히 다른 기억 공간이라 이전 정보를 못본다.

**해결: `/memories/` 경로 라우팅**

`CompositeBackend`로 "특정 경로에 저장된 파일은 모든 대화가 공유"하게 만든다. (#Composit: 여러개를 합친)

| 경로 | 어디 저장? | 다른 대화에서 보이나? |
|---|---|---|
| `/scratch.txt` (그 외 경로) | StateBackend (현재 대화) | N |
| `/memories/me.txt` | StoreBackend (공용 저장소) | Y |

규칙은 **단 하나**: 에이전트가 `/memories/` 폴더에 파일로 저장한 정보들만 다른 대화에서도 기억한다.

→ 아래 셀에서 직접 확인한다.

In [91]:
### 장기메모리 실습
from deepagents.backends import StoreBackend, CompositeBackend
from langgraph.store.memory import InMemoryStore
from langgraph.checkpoint.memory import MemorySaver

# 1) 공용 저장소 (모든 대화가 공유) ─ 운영 시 PostgresStore로 교체, MySQL용은 내장함수 없음.
mem_store = InMemoryStore()
mem_checkpointer = MemorySaver()

# 2) 경로 라우팅 백엔드: /memories/ 만 공용, 나머지는 대화별 격리
def memory_only_backend(runtime):
    return CompositeBackend(   # 여기 중요!
        default=FilesystemBackend(root_dir="./", virtual_mode=True),
        routes={"/memories/": StoreBackend(runtime)},
    )

# 3) 메모리 전용 에이전트 (스킬/도구 없음 — 메모리 동작만 보기 위함)
memory_agent = create_deep_agent(
    model=model,
    system_prompt=(
        "당신은 사용자 정보를 잘 기억하는 어시스턴트입니다.\n\n"
        "## 사용자 식별 규칙\n"
        "- 사용자가 이름을 알려주면 영어 소문자 + 언더스코어로 변환해 user_id로 사용하세요.\n"
        "  예) '김순주' → 'kim_soonju' / '정순주' → 'jeong_soonju' / 'Alice Park' → 'alice_park'\n"
        "- 모든 메모리 파일은 반드시 `/memories/<user_id>/` 폴더 아래에 저장하세요.\n"
        "  예) /memories/kim_soonju/profile.txt, /memories/jeong_soonju/preferences.txt\n"
        "- 절대로 /memories/ 바로 아래에 파일을 만들지 마세요. 반드시 user_id 폴더를 거쳐야 합니다.\n\n"
        "## 저장 규칙\n"
        "- 사용자가 알려준 정보(이름, 선호도, 습관 등)는 즉시 해당 user_id 폴더에 파일로 저장하세요.\n"
        "- 같은 user_id의 기존 파일은 새 정보로 업데이트해도 되지만, **다른 user_id의 파일을 덮어쓰면 안 됩니다.**\n\n"
        "## 조회 규칙\n"
        "- 사용자가 누군가에 대해 물으면, 먼저 이름을 user_id로 변환한 뒤 `/memories/<user_id>/` 폴더의 파일들을 읽어 답하세요.\n"
        "- 여러 사람에 대한 질문이면 각 user_id 폴더를 순차적으로 모두 확인하세요.\n"
        "- 해당 폴더가 없으면 '저장된 정보가 없습니다'라고 답하세요. 추측하지 마세요."
        "이미 파일이 있으면 반드시 먼저 읽어서 기존 내용을 모두 보존한 채 새 줄을 추가하라. "
    "절대 기존 내용을 삭제하거나 덮어쓰지 마라."
    ),

    backend=memory_only_backend,
    store=mem_store,
    checkpointer=mem_checkpointer,
)

In [92]:
# 세션아이디가 달라도 대화를 기억하는지 테스트
r1 = memory_agent.invoke(
    {"messages": [{"role": "user", "content":
        "내 이름은 김순주, 좋아하는 색은 파랑이야. 기억해줘."}]},
    config={"configurable": {"thread_id": "zaq-1"}},
)
print(r1["messages"][-1].content)
print()

기억했어요.



In [93]:
r2 = memory_agent.invoke(
    {"messages": [{"role": "user", "content":
        "내 이름은 정순주, 좋아하는 색은 라벤더야. 기억해줘."}]},
    config={"configurable": {"thread_id": "zaq-2"}},
)
print(r2["messages"][-1].content)
print()

기억해둘게요.  
이름: 정순주  
좋아하는 색: 라벤더



In [94]:
r3 = memory_agent.invoke(
    {"messages": [{"role": "user", "content":
        "김순주랑 정순주가 각각 좋아하는 색이 뭐였지?"}]},
    config={"configurable": {"thread_id": "zaq-3"}},
)
print(r3["messages"][-1].content)

김순주는 파랑, 정순주는 라벤더를 좋아해.


`thread_id`가 완전히 바뀌었는데도 대화 2에서 이름과 색을 기억한다. 그 이유는,

- 대화 1: 에이전트가 `/memories/profile.txt`에 정보를 저장 → `CompositeBackend`가 이 경로를 `StoreBackend`로 라우팅 → `mem_store`(공용 저장소)에 들어감
- 대화 2: 새 thread지만 **같은 `mem_store`를 공유** → 같은 파일을 읽을 수 있다

만약 `/profile.txt` (memories 폴더 밖)에 저장했다면 대화 1에서만 보이고 대화 2에선 사라졌을 것. **`/memories/` 경로 규칙**이 핵심.



## 6. AGENTS.md — 항상 적용되는 "상시 규칙" 파일

지금까지 만든 SKILL.md는 **필요할 때만** 발동된다. 그런데 회사에서는 "어떤 작업을 하든 무조건 지켜야 하는" 규칙이 있다 — 한국어로 답하기, 코딩 컨벤션, 보안 원칙 같은 거요.

이걸 위한 게 **AGENTS.md** 이다. (Claude Code의 `CLAUDE.md`와 같은 개념)

### 시스템 프롬프트랑 뭐가 다른가?

본질은 **같음**. AGENTS.md 내용은 시스템 프롬프트에 `<agent_memory>` 태그로 자동 합쳐져서 LLM에게 전달돼요. 그럼 왜 따로 만드냐?

| | 시스템 프롬프트 | AGENTS.md |
|---|---|---|
| 위치 | 파이썬 코드 안 | 별도 마크다운 파일 |
| 수정 | 개발자가 코드 변경 | 누구나 메모장으로 |
| 공유 | 에이전트마다 따로 | 여러 에이전트가 한 파일 공유 |
| 자가 수정 |  |  에이전트가 스스로 업데이트 가능 |

### SKILL.md vs AGENTS.md

| | AGENTS.md | SKILL.md |
|---|---|---|
| 로딩 | **항상** (매 호출) | **필요할 때만** |
| 용도 | "절대 규칙" | "특정 작업 절차" |
| 크기 | 짧게 | 길어도 OK |
| 비유 | 회사 취업규칙 | 업무 매뉴얼 |

### 실전 패턴

```python
agent = create_deep_agent(
    model=model,
    backend=...,
    memory=["/AGENTS.md"],   # 항상 적용 (상시 규칙)
    skills=["/skills/"],     # 필요 시 발동 (작업 절차)
    subagents=[...],         # 도메인별 전문가
)
```

이렇게 하면:
- **모든 응답**에 AGENTS.md의 회사 규칙이 자동 반영
- **코드 리뷰 요청**이 들어오면 추가로 SKILL.md가 발동
- **세부 영역**은 서브에이전트가 위임받아 처리

→ 결과적으로 한 에이전트가 **상시 규칙 + 작업 절차 + 도메인 분업**을 모두 갖춥니다.

In [95]:
# AGENTS.md 생성 — 포스코 DX팀 상시 규칙
agents_md = """# 포스코 DX팀 에이전트 상시 규칙

## 언어
- 변경 요약은 항상 영어로 말한다.
- 코드 주석은 한국어로 단다.

## 코딩 컨벤션
- Python: Black 포맷터, 타입 힌트 필수
- 변수/함수: snake_case
- 클래스: PascalCase
- 함수는 50줄 이내로 짧게

## 보안 (위반 시 critical)
- API 키, 비밀번호, 토큰을 코드에 절대 박지 않는다.
- DB 쿼리는 반드시 파라미터 바인딩을 사용한다 (f-string 금지).
- 사용자 입력을 시스템 명령에 직접 넣지 않는다.

## 응답 형식
- 가장 하단에 '소리없이 세상을 움직입니다, POSCO' 라는 문구를 넣는다.
- 추측하는 부분은 반드시 "확실하지 않음"이라고 명시한다.
- 코드 변경을 제안할 때는 변경 이유를 한 줄로 함께 적는다.
- 결과 보고는 결론 → 근거 → 권장사항 순서로 작성한다.
"""

with open("./AGENTS.md", "w", encoding="utf-8") as f:
    f.write(agents_md)

print("AGENTS.md 생성 완료: ./AGENTS.md")

AGENTS.md 생성 완료: ./AGENTS.md


In [96]:
# 풀세트 에이전트: AGENTS.md(상시) + SKILL.md(작업) 
full_agent = create_deep_agent(
    model=model,
    system_prompt=(
        "당신은 포스코 DX팀의 코드 리뷰 오케스트레이터입니다. "
        "**반드시 먼저 /skills/github-commit-review/SKILL.md 파일을 읽고** "
        "그 절차를 그대로 따라 리뷰하세요. 매뉴얼을 읽지 않고 답하면 실패입니다."
        "결과를 종합 보고서로 묶어 출력하세요."
    ),
    backend=FilesystemBackend(root_dir="./", virtual_mode=True),
    memory=["/AGENTS.md"],                           # 상시 규칙
    skills=["/skills/github-commit-review/"],        # 작업 절차
)

print("풀세트 에이전트 생성 완료 (memory + skills)")

풀세트 에이전트 생성 완료 (memory + skills)


In [97]:
# 같은 코드를 풀세트 에이전트로 리뷰 — AGENTS.md 규칙(한국어/형식)이 자동 적용됨
result = full_agent.invoke(
    {"messages": [{"role": "user", "content": new_commit_diff}]}
)

In [98]:
msgs = result["messages"]
used_skills = extract_used_skills_v2(msgs)
print("=" * 60)
print("[full_agent 결과 (AGENTS.md 적용)]")
print("=" * 60)
print("발동된 SKILL :", used_skills if used_skills else "(없음)")
print("-" * 60)
print(msgs[-1].content)


[full_agent 결과 (AGENTS.md 적용)]
발동된 SKILL : ['github-commit-review']
------------------------------------------------------------
## 변경 요약
이번 커밋은 사용자 생성, 내보내기, 프로필 조회, 로그 처리 기능을 추가한 것으로 보입니다. 다만 코드 전반에 보안 취약점, 자원 누수, 성능 문제, 컨벤션 위반이 함께 포함되어 있어 기능 추가보다는 즉시 수정이 필요한 상태입니다.

## 발견사항
- [critical] 하드코딩된 시크릿/API 키 — `DB_PASSWORD`, `API_KEY`가 코드에 그대로 노출되어 있습니다. 저장소 유출 시 즉시 악용될 수 있습니다.
- [critical] SQL 인젝션 — `create_user()`와 `get_all_user_profiles()`에서 f-string으로 SQL을 구성하고 있어 사용자 입력으로 쿼리가 변조될 수 있습니다.
- [critical] 시스템 명령 주입 가능성 — `export_user()`에서 `os.system(f"cat {path}")`를 사용해 외부 입력 `user_id`가 경로 조작/명령 실행으로 이어질 수 있습니다.
- [major] DB 연결 누수 — `create_user()`와 `get_all_user_profiles()`에서 `conn.close()`가 없고 예외 처리도 없어 커넥션이 누적될 수 있습니다.
- [major] 외부 API 인증 방식 오류 가능성 — `Authorization` 헤더에 원문 키를 직접 넣고 있어 일반적인 Bearer 스킴을 따르지 않으며, 키 노출 범위도 커집니다. 정확한 API 규격은 **확실하지 않음**.
- [major] 응답/예외 처리 부재 — `requests.get()` 실패, `r.json()` 파싱 실패, `cur.fetchone()`가 `None`인 경우를 처리하지 않아 런타임 오류가 날 수 있습니다.
- [major] 불필요한 성능 저하

** AGENT.md의 타입힌트 필수구문, 마지막문구 반영됨. **

In [99]:
# ### SKILL.md만 연결했을때와 비교
# 발동된 스킬: ['github-commit-review']
# ============================================================
# 변경 요약
# `app/users.py`는 사용자 생성, 내보내기, 프로필 조회, 로그 처리 기능을 한 파일에 담고 있습니다. 신규 커밋으로 보이며, 사용자 입력을 DB 쿼리와 시스템 명령, 외부 API 호출에 직접 연결하는 형태입니다. 의도는 사용자 관리 유틸리티 추가로 추정되지만, 보안 취약점과 자원 정리 누락, 비효율적인 로직이 함께 들어가 있습니다.

# 발견사항
# - [critical] SQL 인젝션 — `create_user()`와 `get_all_user_profiles()`가 f-string으로 SQL을 구성합니다. `name`, `email`, `role`, `uid`가 그대로 쿼리에 들어가며 DB 탈취/변조가 가능합니다.
# - [critical] 하드코딩된 비밀번호/API 키 — `DB_PASSWORD`, `API_KEY`가 소스에 평문으로 포함되어 있습니다. 유출 시 즉시 악용될 수 있습니다.
# - [critical] 명령어 주입 가능성 — `export_user()`가 `os.system(f"cat {path}")`를 사용합니다. `user_id`로 경로가 조작되면 임의 명령 실행으로 이어질 수 있습니다.
# - [major] 데이터베이스 커넥션 미종료 — `create_user()`와 `get_all_user_profiles()`에서 `conn.close()`가 호출되지 않아 연결 누수 및 잠금 문제가 생길 수 있습니다.
# - [major] 예외 처리 누락 — DB 작업, 네트워크 요청, 파일 읽기 모두 실패 가능성이 있는데 처리 로직이 없습니다. 중간 실패 시 리소스 누수와 부분 실패가 발생합니다.
# - [major] 외부 API 인증 방식 부적절 — `Authorization` 헤더에 원시 키만 넣고 있습니다. 보통 `Bearer <token>` 형식이 필요합니다. 또한 응답 상태 검증이 없습니다.
# - [major] `calc()`의 O(n²) 중복 비교 — 전체 로그를 이중 반복으로 비교해 성능이 매우 나쁩니다. 입력이 커지면 심각하게 느려집니다.
# - [major] 파일 핸들 누락 — `open("big_log.txt").readlines()`는 context manager 없이 사용되어 파일이 명확히 닫히지 않습니다.
# - [minor] `calc()`의 인자 `d` 미사용 — 함수 시그니처와 구현이 불일치합니다.
# - [minor] `return x|` 는 문법 오류입니다. 현재 코드 그대로는 실행 자체가 불가능합니다.
# - [nit] `import sqlite3, os, requests, time`는 한 줄 import로 가독성이 떨어집니다.
# - [nit] 함수명 `calc`는 역할이 불명확합니다. 의도를 드러내는 이름이 필요합니다.

# 권장사항
# 1. SQL은 반드시 파라미터 바인딩을 사용하고, `export_user()`는 `os.system` 대신 파일 API로 처리하세요.
# 2. 시크릿은 환경변수나 비밀 관리 시스템으로 옮기고, 커넥션/파일은 `with` 문으로 관리하세요.
# 3. `calc()`는 중복 제거가 목적이면 `set` 또는 `collections.Counter` 기반으로 O(n) 수준으로 개선하세요.
# 4. 외부 API 호출은 타임아웃, 상태 코드 검사, 예외 처리를 추가하세요.
# 5. `return x|` 문법 오류를 수정하고, 미사용 인자와 불명확한 이름을 정리하세요.

# 종합 의견
# 수정 필요

## 핵심 정리

### 핵심 구성요소

| # | 구성요소 | 파일/파라미터 | 로딩 시점 | 용도 |
|---|---|---|---|---|
| 1 | **`@tool`** | 함수 데코레이터 | 호출 시 | 파이썬 함수를 LLM이 쓸 수 있는 도구로 변환 |
| 2 | **`create_deep_agent`** | — | — | 모델/도구/스킬/메모리를 하나로 묶다 |
| 3 | **`SKILL.md`** | `skills=["/skills/"]` | **필요할 때만** | 작업별 절차서 (Progressive Disclosure) |
| 4 | **`AGENTS.md`** | `memory=["/AGENTS.md"]` | **항상** | 상시 규칙·컨벤션 (시스템 프롬프트에 자동 주입) |
| 5 | **장기 메모리** | `CompositeBackend` + `/memories/` | **대화 사이 영속** | 다른 thread에서도 정보 공유 |

### AGENTS.md vs SKILL.md

|  | AGENTS.md | SKILL.md |
|---|---|---|
| 로딩 | 항상 (매 호출) | 필요 시 (LLM이 판단) |
| 토큰 비용 | 매 호출 소비 | 발동될 때만 |
| 크기 | 짧게 | 길어도 OK |
| 개수 | 1~2개 | 수십 개 가능 |
| 본질 | 시스템 프롬프트 연장 | 별도 절차서 |
| 비유 | 회사 취업규칙 | 업무 매뉴얼 |

### SKILL.md 작성 규칙

```yaml
---
name: <소문자-하이픈, 64자 이내>          # 식별자
description: <한 문장, 1024자 이내>        #  LLM 매칭에 사용 — 가장 중요
allowed-tools: tool1, tool2                # (선택) 도구 제한
---

# 본문
## 사용 시기
## 절차
## 출력 형식
## 예외 처리
```

### 장기 메모리 핵심 규칙

- `/memories/*` 경로에 저장한 파일만 영구 저장된다
- `thread_id`가 달라도 같은 파일에 접근 가능
- 개발: `InMemoryStore` → 운영: `PostgresStore`로 교체

### 풀세트 패턴 (회사용 표준 형태)

```python
agent = create_deep_agent(
    model=model,
    system_prompt="...",
    tools=[find_employee, ...],          # 1. @tool
    backend=composite_backend,                # 5. 장기 메모리 백엔드
    store=store,
    checkpointer=checkpointer,
    memory=["/AGENTS.md"],                    # 4. 상시 규칙
    skills=["/skills/"],                      # 3. 작업 절차
)
```

### 실무 가이드

- **항상 적용되는 규칙** → `AGENTS.md`
- **특정 작업 절차** → `SKILL.md`
- **사용자별 영속 정보** → 장기 메모리 (`/memories/`)
- **여러 스킬 운영 시**: `description`을 잘 써야 LLM이 올바른 스킬을 자동 선택한다